# Awaaz — Colab Data Preparation Runner

This notebook automates downloading bilingual speech corpora (English and Punjabi), converting to standardized 16 kHz mono FLAC, enforcing duration and sacred-text policies, calculating balanced manifests, archiving data to Google Drive, and generating global CMVN statistics and the SentencePiece bilingual tokenizer.

In [ ]:
# 1. Mount Google Drive for persistent artifact storage
from google.colab import drive

drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/awaaz/data
!mkdir -p /content/drive/MyDrive/awaaz/artifacts
!mkdir -p /content/drive/MyDrive/awaaz/runs

In [ ]:
# 2. Read Secrets (Hugging Face Token) securely via Colab Secrets
import os

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN successfully configured from Colab Secrets.')
    else:
        print('HF_TOKEN not found in userdata. Public datasets will be used.')
except Exception as e:
    print(f'Secrets access note: {e}')

In [ ]:
# 3. Clone Repository & Install Package Dependencies
%cd /content
!git clone https://github.com/Paramveersingh-S/Awaaz.git repo
%cd /content/repo
!git checkout main
!pip install -q --upgrade pip
!pip install -e .[pipeline,export]

In [ ]:
# 4. Ingest and Prepare Datasets (FLEURS & Common Voice)
# Prepares 16 kHz mono FLAC audio files and manifests under /content/drive/MyDrive/awaaz/data
!python scripts/prepare_hf.py \
  --dataset_id google/fleurs \
  --subset en_us \
  --language en \
  --split train \
  --output_dir /content/drive/MyDrive/awaaz/data/fleurs_en

!python scripts/prepare_hf.py \
  --dataset_id google/fleurs \
  --subset pa_in \
  --language pa \
  --split train \
  --output_dir /content/drive/MyDrive/awaaz/data/fleurs_pa

In [ ]:
# 5. Merge Manifests with Language Balancing (50/50 English-Punjabi)
!python scripts/merge_manifests.py \
  --manifests /content/drive/MyDrive/awaaz/data/fleurs_en/train_manifest.jsonl /content/drive/MyDrive/awaaz/data/fleurs_pa/train_manifest.jsonl \
  --languages en pa \
  --repeats 1 1 \
  --output /content/drive/MyDrive/awaaz/data/train_combined.jsonl

In [ ]:
# 6. Package Audio Dataset Archive with Checksum
# Compresses audio to a tar archive on Google Drive for rapid single-file transfer to local NVMe during training
%cd /content/drive/MyDrive/awaaz/data
!tar -cf train_audio.tar fleurs_en/audio fleurs_pa/audio
!sha256sum train_audio.tar > train_audio.tar.sha256
print('Dataset archived successfully on Google Drive with SHA-256 checksum.')

In [ ]:
# 7. Train Bilingual Tokenizer & Compute Global CMVN
%cd /content/repo
# Bilingual SentencePiece Unigram Tokenizer (vocab=1024, blank=0, unk=1)
!python -m awaaz.text train-tokenizer \
  --manifests /content/drive/MyDrive/awaaz/data/train_combined.jsonl \
  --output /content/drive/MyDrive/awaaz/artifacts/tokenizer \
  --vocab_size 1024

# Global Cepstral Mean and Variance Normalization (80-bin mel)
!python -m awaaz.features compute-cmvn \
  --manifest /content/drive/MyDrive/awaaz/data/train_combined.jsonl \
  --output /content/drive/MyDrive/awaaz/artifacts/cmvn.npz

print('Data preparation complete. Artifacts ready for training.')